# NadiKampus — Pelabelan Sentimen Jawaban GForm

Notebook ini membuat **label sentimen (positif / netral / negatif)** untuk jawaban singkat survei, lalu menggabungkannya dengan label Telegram yang sudah ada menjadi satu dataset latih.

**Kenapa semua jawaban dilabeli, bukan sampel?** Jawaban survei hanya sekitar 550-630 baris (5 pertanyaan terbuka × responden) dan pendek, jadi melabeli semuanya realistis (kira-kira 1 jam). Dataset sekecil ini tidak perlu "ditebak" oleh model sebelum dilabeli manusia.

**Alur:** `muat jawaban → beri label (UI di notebook / Excel) → cek sebaran → hitung kesepakatan antar-pelabel → gabung dengan label Telegram → simpan + split`

### Panduan label (pakai aturan yang sama untuk semua pelabel)

Nilai **kondisi yang diungkapkan jawaban**, dengan mempertimbangkan pertanyaannya. Jawaban singkat seperti "orang tua" tidak bisa dinilai tanpa tahu pertanyaannya.

| Pertanyaan | Positif | Netral | Negatif |
|---|---|---|---|
| **Yang membuat bertahan** / **yang membuat diterima** (`wellbeing`, `social`) | menyebut sumber dukungan ("orang tua", "teman yang peduli") | jawaban datar tanpa makna jelas | merasa tidak ada pendukung / tidak diterima |
| **Kendala akademik** / **kesulitan ekonomi** (`academic`, `economic`) | menyatakan baik-baik saja | menyebut topik tanpa nada keluhan | menyebut kendala atau kesulitan ("tugas menumpuk", "UKT mahal") |
| **Dukungan karier yang dibutuhkan** (`career`) | puas dengan dukungan yang ada | permintaan datar ("pelatihan", "info magang") | cemas / merasa kurang ("bingung mulai dari mana") |

Aturan tambahan:

* **Kalimat campuran (memuji dulu, lalu mengkritik dengan "tetapi / namun / sayangnya")** → nilai **maksud utamanya**, yang biasanya ada *setelah* kata pertentangan. Bagian setelahnya berisi keluhan atau kekurangan → **Negatif** ("Fasilitas bagus, tetapi AC lab sering mati"). Bagian setelahnya hanya harapan atau saran tanpa keluhan → **Netral** ("Bebannya berat, tapi masih bisa saya atur kalau jadwal dibuat lebih tersebar").
* **Negasi membalik makna**: "tidak ramah", "kurang jelas", "belum membantu" → **Negatif**. Perhatikan juga pertanyaannya: "tidak ada kendala" adalah kondisi baik.
* **Netral hanya untuk teks tanpa sikap**: pertanyaan, permintaan info, atau fakta datar. Jangan memakai Netral sebagai tempat menaruh jawaban yang sulit diputuskan.
* Jawaban yang campur tanpa kata pertentangan → pilih nada yang dominan. Ragu → pilih label terbaik lalu **centang "Tandai ragu"** (kasus ragu dibahas bersama, sesuai Pedoman proyek). Jangan menebak berdasarkan identitas atau prodi.


## 1. Setup & konfigurasi

In [1]:
# Jalankan sekali kalau belum terpasang, lalu restart kernel:
# %pip install pandas pyarrow ipywidgets openpyxl scikit-learn

In [2]:
import os, html
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

In [3]:
# ====================== KONFIGURASI ======================
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), Path.cwd())
CORPUS_PATH = next((p for p in [ROOT / "data" / "processed" / "nadikampus_corpus_all.parquet",
                                ROOT / "data" / "nadikampus_corpus_all.parquet",
                                ROOT / "nadikampus_corpus_all.parquet",
                                Path.cwd() / "nadikampus_corpus_all.parquet"] if p.exists()),
                   ROOT / "data" / "processed" / "nadikampus_corpus_all.parquet")
OUT_DIR = ROOT / "data" / "labeling"            # jangan di-commit ke GitHub (berisi teks jawaban)

ANNOTATOR = "annotator1"      # <- isi namamu. Tiap pelabel punya file sendiri: labels_gform_<nama>.parquet
SEED = 42                     # semua pelabel harus memakai SEED yang sama agar urutan soalnya identik
ONLY_VALID = True             # hanya jawaban is_valid (bukan "tidak ada" / kosong makna)
SHOW_OTHER_ANSWERS = False    # True = tampilkan jawaban lain dari responden yang sama (membantu, tapi bisa membiaskan)

SENT_LABELS  = ["negative", "neutral", "positive"]
SENT_DISPLAY = {"negative": "😠 Negatif", "neutral": "😐 Netral", "positive": "🙂 Positif"}
DIM_QUESTION = {
    "wellbeing": "Hal apa yang paling membuat Anda tetap bertahan …",
    "academic":  "Apa kendala akademik yang paling sering Anda alami …",
    "social":    "Apa yang membuat Anda merasa lebih diterima …",
    "career":    "Dukungan apa yang paling Anda butuhkan untuk karier …",
    "economic":  "Ceritakan singkat kesulitan ekonomi yang Anda alami …",
}

## 2. Muat jawaban GForm

In [4]:
assert CORPUS_PATH.exists(), f"File korpus tidak ditemukan: {CORPUS_PATH.resolve()} -> ubah CORPUS_PATH"
corpus = pd.read_parquet(CORPUS_PATH)

gf = corpus[corpus["source"].astype(str) == "gform"].copy()
if ONLY_VALID:
    gf = gf[gf["is_valid"]]
gf["dimension"] = gf["dimension"].astype(str)
gf = gf.sort_values("doc_id").sample(frac=1, random_state=SEED).reset_index(drop=True)   # acak, tetap sama antar pelabel

print(f"Jawaban GForm yang akan dilabeli: {len(gf)}  (dari {int((corpus['source'].astype(str) == 'gform').sum())} jawaban)")
gf.groupby("dimension").agg(n=("doc_id", "size"), median_kata=("n_words", "median")).T

Jawaban GForm yang akan dilabeli: 553  (dari 626 jawaban)


dimension,academic,career,economic,social,wellbeing
n,118.0,131.0,29.0,132.0,143.0
median_kata,10.0,6.0,14.0,4.5,4.0


## 3. Penyimpanan label

Setiap klik langsung tersimpan ke `data/labeling/labels_gform_<ANNOTATOR>.parquet` (ditulis aman: file sementara lalu diganti). Menutup dan membuka lagi notebook **tidak menghilangkan label**; antrean hanya berisi jawaban yang belum dilabeli.

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
STORE_PATH = OUT_DIR / f"labels_gform_{ANNOTATOR}.parquet"
STORE_COLS = ["doc_id", "label_sentiment", "ragu", "labeled_at", "annotator"]

def load_store():
    if STORE_PATH.exists():
        return pd.read_parquet(STORE_PATH)
    return pd.DataFrame({c: pd.Series(dtype="object") for c in STORE_COLS})

def save_store(store):
    tmp = STORE_PATH.with_suffix(".tmp")
    store.to_parquet(tmp, index=False)
    os.replace(tmp, STORE_PATH)

def upsert(store, doc_id, sentiment, ragu=False):
    row = pd.DataFrame([{"doc_id": doc_id, "label_sentiment": sentiment, "ragu": bool(ragu),
                         "labeled_at": datetime.now().isoformat(timespec="seconds"), "annotator": ANNOTATOR}])
    store = store[store["doc_id"] != doc_id]
    return pd.concat([store, row], ignore_index=True) if len(store) else row

store = load_store()
print(f"Label tersimpan untuk '{ANNOTATOR}': {len(store)} dari {len(gf)} jawaban")

class LabelSession:
    """Logika antrean labeling (dipisah dari UI supaya mudah diuji)."""
    def __init__(self, rows, store):
        self.rows = rows.set_index("doc_id", drop=False)
        self.store = store
        done = set(store["doc_id"])
        self.queue = [d for d in rows["doc_id"] if d not in done]
        self.history = []

    def current(self):
        return self.queue[0] if self.queue else None

    def existing(self, doc_id):
        r = self.store[self.store["doc_id"] == doc_id]
        return (r["label_sentiment"].iloc[0], bool(r["ragu"].iloc[0])) if len(r) else (None, False)

    def label(self, sentiment, ragu=False):
        doc = self.queue.pop(0)
        self.store = upsert(self.store, doc, sentiment, ragu)
        save_store(self.store)
        self.history.append(doc)

    def skip(self):
        if len(self.queue) > 1:
            self.queue.append(self.queue.pop(0))

    def back(self):
        if self.history:
            self.queue.insert(0, self.history.pop())

    def done_count(self):
        return int(self.rows.index.isin(self.store["doc_id"]).sum())

    def total(self):
        return len(self.rows)

Label tersimpan untuk 'annotator1': 0 dari 553 jawaban


## 4. Labeling di notebook (UI interaktif)

Jalankan sel di bawah, lalu **satu klik pada tombol sentimen langsung menyimpan dan pindah** ke jawaban berikutnya.

* **Tandai ragu** dicentang *sebelum* mengklik tombol sentimen, untuk kasus ambigu yang perlu dibahas.
* **Kembali** membuka lagi jawaban yang baru dilabeli (label lama terisi; klik tombol lain untuk mengubah).
* **Lewati** memindahkan jawaban ke akhir antrean.

Kalau widget tidak tampil (kadang terjadi di editor tertentu), pakai jalur Excel di bagian 5.

In [6]:
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets belum terpasang -> jalankan:  %pip install ipywidgets   (atau pakai jalur Excel di bagian 5)")

def _esc(t):
    return html.escape(str(t)).replace("\n", "<br>")

# jawaban lain dari responden yang sama (opsional)
OTHERS = {}
for r in gf.itertuples():
    OTHERS.setdefault(r.author_id, []).append((r.dimension, r.text_raw))

class LabelUI:
    SEL_STYLE = {"negative": "danger", "neutral": "info", "positive": "success"}

    def __init__(self, session):
        self.s = session
        self.sent = None
        self.card, self.prog = W.HTML(), W.HTML()
        self.sent_btn = {k: W.Button(description=SENT_DISPLAY[k], layout=W.Layout(width="150px", height="46px"))
                         for k in SENT_LABELS}
        for k, b in self.sent_btn.items():
            b.on_click(lambda _b, k=k: self._pick(k))
        self.ragu = W.Checkbox(value=False, description="Tandai ragu / ambigu", indent=False)
        self.btn_skip = W.Button(description="Lewati", icon="forward")
        self.btn_back = W.Button(description="Kembali", icon="backward")
        self.btn_skip.on_click(lambda _b: (self.s.skip(), self._render()))
        self.btn_back.on_click(lambda _b: (self.s.back(), self._render()))
        self.box = W.VBox([self.prog, self.card, W.HBox(list(self.sent_btn.values())), self.ragu,
                           W.HBox([self.btn_back, self.btn_skip])])
        self._render()

    def _style(self):
        for k, b in self.sent_btn.items():
            b.button_style = self.SEL_STYLE[k] if k == self.sent else ""

    def _pick(self, k):                     # satu klik = simpan + lanjut
        if self.s.current() is None:
            return
        self.s.label(k, bool(self.ragu.value))
        self._render()

    def _render(self):
        done, total = self.s.done_count(), self.s.total()
        self.prog.value = (f"<b>Terlabel {done}/{total}</b> ({done / max(total, 1):.0%}) · "
                           f"antrean tersisa {len(self.s.queue)} · pelabel: <code>{html.escape(ANNOTATOR)}</code>")
        doc = self.s.current()
        if doc is None:
            self.card.value = "<h3>Selesai 🎉 Semua jawaban sudah dilabeli.</h3>"
            self.sent, self.ragu.value = None, False
            self._style()
            return
        r = self.s.rows.loc[doc]
        extra = ""
        if SHOW_OTHER_ANSWERS:
            oth = [f"<li><i>{html.escape(d)}</i>: {_esc(t)}</li>" for d, t in OTHERS.get(r["author_id"], []) if t != r["text_raw"]]
            if oth:
                extra = ("<div style='font-size:12px;opacity:.7;margin-top:10px'>Jawaban lain responden ini:</div>"
                         f"<ul style='font-size:13px;opacity:.85;margin:4px 0 0 18px'>{''.join(oth)}</ul>")
        self.card.value = (
            "<div style='padding:12px 14px;border:1px solid rgba(128,128,128,.35);border-radius:8px;"
            "background:rgba(128,128,128,.08);max-width:760px'>"
            f"<div style='font-size:11px;opacity:.6;margin-bottom:6px'>{html.escape(r['dimension'])} · {html.escape(doc)}</div>"
            f"<div style='font-size:12px;opacity:.7;margin-bottom:6px'>Pertanyaan: {_esc(DIM_QUESTION.get(r['dimension'], ''))}</div>"
            f"<div style='font-size:17px;line-height:1.5'>{_esc(r['text_raw'])}</div>{extra}</div>")
        old_sent, old_ragu = self.s.existing(doc)       # terisi saat membuka ulang lewat 'Kembali'
        self.sent = old_sent if old_sent in SENT_LABELS else None
        self.ragu.value = old_ragu
        self._style()

if HAVE_WIDGETS:
    session = LabelSession(gf, store)
    ui = LabelUI(session)
    display(ui.box)

## 5. Alternatif: labeling lewat Excel

1. `export_excel()` membuat `labeling_gform_<ANNOTATOR>.xlsx` berisi jawaban yang belum dilabeli, dengan **dropdown** pada kolom label.
2. Isi `label_sentiment` (opsional `ragu` dengan `x`), simpan.
3. `import_excel()` memasukkan label ke penyimpanan yang sama dengan UI. Baris kosong diabaikan, jadi boleh diisi sebagian.

File tidak ditimpa kalau sudah ada (supaya isian tidak hilang); pakai `overwrite=True` untuk membuat ulang.

In [7]:
def export_excel(overwrite=False):
    from openpyxl.worksheet.datavalidation import DataValidation
    path = OUT_DIR / f"labeling_gform_{ANNOTATOR}.xlsx"
    if path.exists() and not overwrite:
        print(f"{path} sudah ada -> tidak ditimpa. Isi lalu import_excel(), atau export_excel(overwrite=True).")
        return path
    done = set(load_store()["doc_id"])
    todo = gf[~gf["doc_id"].isin(done)]
    out = pd.DataFrame({"doc_id": todo["doc_id"], "dimension": todo["dimension"],
                        "pertanyaan": todo["dimension"].map(DIM_QUESTION), "jawaban": todo["text_raw"],
                        "label_sentiment": "", "ragu": ""})
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, index=False, sheet_name="label")
        ws = xw.sheets["label"]
        dv = DataValidation(type="list", formula1='"negative,neutral,positive"', allow_blank=True)
        ws.add_data_validation(dv)
        col = {n: i + 1 for i, n in enumerate(out.columns)}
        L = lambda name: ws.cell(row=1, column=col[name]).column_letter
        dv.add(f"{L('label_sentiment')}2:{L('label_sentiment')}{len(out) + 1}")
        for name, width in {"doc_id": 14, "dimension": 11, "pertanyaan": 44, "jawaban": 60, "label_sentiment": 16, "ragu": 7}.items():
            ws.column_dimensions[L(name)].width = width
        ws.freeze_panes = "A2"
    print(f"{len(out)} jawaban diekspor ke {path}")
    return path

def import_excel(path=None):
    path = Path(path) if path else OUT_DIR / f"labeling_gform_{ANNOTATOR}.xlsx"
    x = pd.read_excel(path, sheet_name="label")
    x["label_sentiment"] = x["label_sentiment"].astype(str).str.strip().str.lower()
    ok = x[x["label_sentiment"].isin(SENT_LABELS)]
    st = load_store()
    for r in ok.itertuples():
        st = upsert(st, r.doc_id, r.label_sentiment, str(r.ragu).strip().lower() in ("x", "1", "ya", "true"))
    if len(ok):
        save_store(st)
    print(f"{len(ok)} label diimpor dari {len(x)} baris | total tersimpan: {len(st)}")
    return st

# contoh:  export_excel()   ... isi di Excel ...   import_excel()

## 6. Cek sebaran label

Sebaran **per dimensi** wajar timpang: jawaban atas "kendala" hampir pasti banyak yang negatif, jawaban atas "yang membuat bertahan" banyak yang positif. Itu bukan kesalahan, tapi memengaruhi cara model dievaluasi (lihat bagian 8). Perhatikan juga jumlah kasus yang ditandai ragu.

In [8]:
store = load_store()
lab = store.merge(gf[["doc_id", "dimension"]], on="doc_id", how="left")
print(f"Terlabel: {len(lab)} / {len(gf)} | ditandai ragu: {int(lab['ragu'].sum()) if len(lab) else 0}\n")
if len(lab):
    ct = pd.crosstab(lab["dimension"], lab["label_sentiment"]).reindex(columns=SENT_LABELS, fill_value=0)
    ct["total"] = ct.sum(axis=1)
    display(ct)
    share = lab["label_sentiment"].value_counts(normalize=True).reindex(SENT_LABELS).fillna(0)
    print("Sebaran keseluruhan:", {k: f"{v:.0%}" for k, v in share.items()})
    for k, v in share.items():
        if v < 0.10:
            print(f"[!] Kelas '{k}' hanya {v:.0%} -> sedikit untuk pelatihan model.")

Terlabel: 0 / 553 | ditandai ragu: 0



## 7. Kesepakatan antar-pelabel (sangat disarankan untuk laporan)

Minta minimal satu orang lain melabeli **sekurang-kurangnya 100-150 jawaban yang sama** (ia menjalankan notebook ini dengan `ANNOTATOR` berbeda; file `labels_gform_<nama>.parquet`-nya ditaruh di `data/labeling/`). Karena urutannya sama (`SEED`), 100-150 jawaban pertama otomatis tumpang tindih. Pedoman: **Cohen's κ ≥ 0,6** = kesepakatan substansial; di bawah 0,4 berarti panduan label perlu diperjelas.

In [9]:
files = sorted(OUT_DIR.glob("labels_gform_*.parquet"))
stores = {f.stem.replace("labels_gform_", ""): pd.read_parquet(f).set_index("doc_id") for f in files}
print("Pelabel ditemukan:", {k: len(v) for k, v in stores.items()})

if len(stores) < 2:
    print("\nBelum ada pelabel kedua -> lewati.")
else:
    names, rows = list(stores), []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = stores[names[i]], stores[names[j]]
            common = a.index.intersection(b.index)
            if len(common) < 30:
                continue
            ya, yb = a.loc[common, "label_sentiment"], b.loc[common, "label_sentiment"]
            rows.append({"pelabel_A": names[i], "pelabel_B": names[j], "n_overlap": len(common),
                         "persen_sama": round(float((ya == yb).mean()), 3),
                         "kappa": round(float(cohen_kappa_score(ya, yb)), 3)})
    display(pd.DataFrame(rows) if rows else "Overlap antar pelabel < 30 jawaban.")

Pelabel ditemukan: {}

Belum ada pelabel kedua -> lewati.


## 8. Gabungkan dengan label Telegram & simpan dataset

Isi `TELEGRAM_LABELS_PATH` dengan file label Telegram-mu (`.parquet`, `.csv`, atau `.xlsx`) dan sesuaikan nama kolomnya. Notebook akan:

* menyeragamkan nama label (`positif/netral/negatif` → `positive/neutral/negative`),
* menggabungkan dengan teks dari korpus lewat `doc_id`,
* untuk GForm, menambahkan **pertanyaan di depan jawaban** (`model_input`), karena jawaban singkat tidak bermakna tanpa pertanyaannya,
* membagi **train / val / test per sumber** (stratifikasi menurut label), jadi performa Telegram dan GForm bisa dilaporkan **terpisah**.

Hasil: `data/processed/nadikampus_sentiment_labeled.parquet`.

In [18]:
MAIN_ANNOTATOR = ANNOTATOR

# 1. Sesuaikan lokasi file & nama kolom Telegram sesuai file CSV asli
TELEGRAM_LABELS_PATH = ROOT / "data" / "processed" / "data_labeled_ags_sep_okt.csv"
# Jika file berada di folder data/processed, gunakan: ROOT / "data" / "processed" / "data_labeled_ags_sep_okt.csv"

TG_ID_COL, TG_LABEL_COL = "Message_ID", "sentiment_label"  # <- SESUAIKAN DI SINI

LABEL_MAP = {
    "positif": "positive", "netral": "neutral", "negatif": "negative",
    "positive": "positive", "neutral": "neutral", "negative": "negative"
}

# --- GForm
g = pd.read_parquet(OUT_DIR / f"labels_gform_{MAIN_ANNOTATOR}.parquet")[["doc_id", "label_sentiment", "ragu"]]
g = g.merge(gf[["doc_id", "source", "doc_type", "dimension", "text_raw", "text_clean", "text_norm", "timestamp"]], on="doc_id")
g["model_input"] = g["dimension"].map({k: v.rstrip(" …") for k, v in DIM_QUESTION.items()}) + ": " + g["text_clean"]

parts = [g]

# --- Telegram
if TELEGRAM_LABELS_PATH:
    p = Path(TELEGRAM_LABELS_PATH)
    t = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_excel(p) if p.suffix in (".xlsx", ".xls") else pd.read_csv(p)
    
    # Filter hanya data Telegram yang relevan (Is_relevant == 'Yes') jika diperlukan
    if "Is_relevant" in t.columns:
        t = t[t["Is_relevant"] == "Yes"].copy()

    ids = t[TG_ID_COL].astype(str).str.strip()
    t["doc_id"] = np.where(ids.str.fullmatch(r"\d+"), "tg_" + ids, ids)             # Message_ID -> doc_id korpus
    t["label_sentiment"] = t[TG_LABEL_COL].astype(str).str.strip().str.lower().map(LABEL_MAP)
    
    bad = t["label_sentiment"].isna().sum()
    if bad:
        print(f"[!] {bad} baris label Telegram tidak dikenali dan dibuang. Nilai asli: {t.loc[t['label_sentiment'].isna(), TG_LABEL_COL].unique()[:8]}")
    
    t = t.dropna(subset=["label_sentiment"])
    tg = t[["doc_id", "label_sentiment"]].drop_duplicates("doc_id").merge(
        corpus[corpus["source"].astype(str) == "telegram"][["doc_id", "source", "doc_type", "dimension", "text_raw", "text_clean", "text_norm", "timestamp"]],
        on="doc_id", how="inner")
    
    tg["ragu"] = False
    tg["model_input"] = tg["text_clean"]
    print(f"Telegram: {len(t)} label di file -> {len(tg)} cocok dengan korpus")
    parts.append(tg)
else:
    print("TELEGRAM_LABELS_PATH kosong -> dataset hanya berisi GForm.")

data = pd.concat(parts, ignore_index=True)
for c in ["source", "doc_type"]:
    data[c] = data[c].astype(str)

# --- Split per sumber
data["split"] = "train"
for src, idx in data.groupby("source").groups.items():
    sub = data.loc[idx]
    strat = sub["label_sentiment"] if sub["label_sentiment"].value_counts().min() >= 5 else None
    if strat is None:
        print(f"[!] {src}: ada kelas < 5 contoh -> split tanpa stratifikasi")
    i_tr, i_rest = train_test_split(sub.index, test_size=0.30, random_state=SEED, stratify=strat)
    rest = data.loc[i_rest, "label_sentiment"]
    i_va, i_te = train_test_split(i_rest, test_size=0.50, random_state=SEED,
                                  stratify=rest if rest.value_counts().min() >= 2 else None)
    data.loc[i_va, "split"], data.loc[i_te, "split"] = "val", "test"

out_path = ROOT / "data" / "processed" / "nadikampus_sentiment_labeled.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(out_path, index=False)
print(f"\n{len(data)} dokumen berlabel -> {out_path}")
display(pd.crosstab([data["source"], data["split"]], data["label_sentiment"]).reindex(columns=SENT_LABELS, fill_value=0))

[!] 1 baris label Telegram tidak dikenali dan dibuang. Nilai asli: <ArrowStringArray>
[nan]
Length: 1, dtype: str
Telegram: 1803 label di file -> 1634 cocok dengan korpus

2187 dokumen berlabel -> c:\Users\yuhanidz\Nadi-Kampus-\data\processed\nadikampus_sentiment_labeled.parquet


label_sentiment  negative  neutral  positive
source   split                              
gform    test          21       14        48
         train         97       66       224
         val           21       14        48
telegram test         102      132        12
         train        474      616        53
         val          102      132        11

### Langkah berikutnya

```python
d = pd.read_parquet("data/processed/nadikampus_sentiment_labeled.parquet")
tr, va, te = (d[d.split == s] for s in ("train", "val", "test"))

# baseline: TF-IDF + Logistic Regression, dilatih di gabungan, DIEVALUASI PER SUMBER
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vec.fit_transform(tr.model_input), tr.label_sentiment)
for src in ["telegram", "gform"]:
    t = te[te.source == src]
    print(src); print(classification_report(t.label_sentiment, clf.predict(vec.transform(t.model_input)), zero_division=0))
```

Laporkan **Macro F1 dan F1 per kelas terpisah untuk Telegram dan GForm**. Sebaran label GForm yang condong menurut dimensi bisa membuat akurasi tampak tinggi padahal model hanya menebak dari pertanyaannya.